# Bronze to Silver — CineData Analytics

**Regra da camada Silver:** as tabelas Bronze não são alteradas em nenhum momento —
elas são apenas lidas. Toda a limpeza, tipagem e renomeação para português acontece aqui,
gerando tabelas novas no database `silver`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

## 1) silver.tb_info_filmes
Origem: `bronze.tb_movies_info`

Tratamentos aplicados:
1. Deduplicação por filme, mantendo a linha de ingestão mais recente.
2. Normalização e tradução da coluna de status.
3. Conversão da data de lançamento testando os 3 formatos presentes na origem.
4. Criação da coluna derivada `ano_lancamento`.

In [0]:
df_bronze_info = spark.table("bronze.tb_movies_info")

print(f"Linhas na Bronze: {df_bronze_info.count()}")
print(f"Filmes distintos: {df_bronze_info.select('id').distinct().count()}")

### 1.1 Deduplicação
A Bronze grava em modo append, então o mesmo filme pode aparecer mais de uma vez
(seja por duplicidade na origem, seja por reexecução do pipeline).
A regra é manter apenas a versão mais recente, e é para isso que existe a coluna
`ingestion_datetime`: ela ordena as versões do mesmo `id`.

In [0]:
janela_filme = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_dedup = (
    df_bronze_info
    .withColumn("num_versao", F.row_number().over(janela_filme))
    .filter(F.col("num_versao") == 1)
    .drop("num_versao")
)

print(f"Linhas após deduplicação: {df_dedup.count()}")

### 1.2 Normalização e tradução do status
A origem traz o mesmo status escrito de várias formas: `Released`, `RELEASED`, `released`,
`In-Production`, `In Production`. Existem ainda registros corrompidos (datas no lugar do status).

Por isso a normalização vem **antes** da tradução: primeiro padronizo tudo para uma forma única
(sem hífen, sem espaço extra, tudo em caixa alta) e só depois traduzo.
Assim o dicionário de tradução precisa de apenas uma entrada por status, e não de uma por variação.

In [0]:
status_normalizado = F.upper(
    F.trim(
        F.regexp_replace(
            F.regexp_replace(F.col("status"), r"[-_]+", " "),  # hífens e underlines viram espaço
            r"\s+", " "                                        # múltiplos espaços viram um só
        )
    )
)

traducao_status = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado",
    "CANCELLED": "Cancelado",   # grafia alternativa, por segurança
}

# Qualquer valor que não esteja no dicionário (corrompido, nulo ou fora do domínio)
# cai no valor padrão "Não Informado".
coluna_status = F.coalesce(
    F.create_map([F.lit(x) for par in traducao_status.items() for x in par])[status_normalizado],
    F.lit("Não Informado")
)

### 1.3 Conversão da data de lançamento
A origem mistura três formatos. Testei cada um nos dados e confirmei qual é qual
verificando se o primeiro campo ultrapassa 12 (se ultrapassa, é dia, não mês):

| Exemplo | Formato |
|---|---|
| `2016-02-09` | `yyyy-MM-dd` |
| `04-25-2018` | `MM-dd-yyyy` |
| `16/03/2017` | `dd/MM/yyyy` |

Uso `try_to_date` dentro de um `coalesce`: ele tenta um formato por vez e só devolve NULL
quando **nenhum** dos três funciona. O `try_` é importante porque a versão comum (`to_date`)
lança exceção em valor malformado e derrubaria o pipeline inteiro por causa de 2 registros ruins.

In [0]:
coluna_data = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
)

### 1.4 Seleção final, renomeação e tipagem
`duracao_minutos` recebe `cast("int")`: por causa do column shift existem registros com texto
no lugar do número, e o cast os transforma em NULL sem quebrar a execução.

In [0]:
df_silver_info = df_dedup.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("title").cast("string").alias("titulo"),
    F.col("original_title").cast("string").alias("titulo_original"),
    coluna_data.alias("data_lancamento"),
    F.col("runtime").cast("int").alias("duracao_minutos"),
    F.col("original_language").cast("string").alias("idioma_original"),
    coluna_status.alias("status_filme"),
    F.col("overview").cast("string").alias("sinopse"),
    F.col("tagline").cast("string").alias("frase_divulgacao"),
).withColumn(
    "ano_lancamento", F.year(F.col("data_lancamento")).cast("int")  # coluna derivada
)

In [0]:
(
    df_silver_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

print("silver.tb_info_filmes gravada.")

### 1.5 Validação

In [0]:
df_check = spark.table("silver.tb_info_filmes")

print(f"Total de linhas: {df_check.count()}")
print(f"Ids distintos:   {df_check.select('id_filme').distinct().count()}  (devem ser iguais)")
df_check.printSchema()

In [0]:
display(df_check.groupBy("status_filme").count().orderBy(F.desc("count")))

In [0]:
print(f"Datas que não puderam ser convertidas: {df_check.filter(F.col('data_lancamento').isNull()).count()}")
display(df_check.select("id_filme", "titulo", "data_lancamento", "ano_lancamento", "status_filme").limit(20))